# Generación de datos para INSERT

Lee los CSV brutos y construye un DataFrame por cada tabla de la BD,
respetando el orden de claves foráneas.

In [3]:
import pandas as pd

DS_PROYECTOS = ['HLF', 'EDA', 'BBDD', 'ML', 'Deployment']
FS_PROYECTOS = ['WebDev', 'FrontEnd', 'Backend', 'React', 'FullStack']

## 1. Lectura de CSVs

In [7]:
clase1   = pd.read_csv('../src/datos_brutos/clase_1.csv', sep=';')
clase2   = pd.read_csv('../src/datos_brutos/clase_2.csv', sep=';')
clase3   = pd.read_csv('../src/datos_brutos/clase_3.csv', sep=';')
clase4   = pd.read_csv('../src/datos_brutos/clase_4.csv', sep=';')
claustro = pd.read_csv('../src/datos_brutos/claustro.csv', sep=';')

DS_RENAME = {f'Proyecto_{p}': p for p in DS_PROYECTOS}
FS_RENAME = {
    'Proyecto_WebDev':    'WebDev',
    'Proyecto_FrontEnd':  'FrontEnd',
    'Proyecto_Backend':   'Backend',
    'Proyecto_React':     'React',
    'Proyecto_FullSatck': 'FullStack',  # typo en el CSV original
}

for df, rename_map, vert in [
    (clase1, DS_RENAME, 'DS'), (clase2, DS_RENAME, 'DS'),
    (clase3, FS_RENAME, 'FS'), (clase4, FS_RENAME, 'FS'),
]:
    df.rename(columns={**rename_map, 'Promoci\u00f3n': 'Promocion'}, inplace=True)
    df['Vertical'] = vert

all_est = pd.concat([clase1, clase2, clase3, clase4], ignore_index=True)
all_est['Fecha_comienzo'] = pd.to_datetime(all_est['Fecha_comienzo'], format='%d/%m/%Y')

all_est.head(3)

,Nombre,Email,Promocion,Fecha_comienzo,Campus,HLF,EDA,BBDD,ML,Deployment,Vertical,WebDev,FrontEnd,Backend,React,FullStack
0,Jafet Casals,Jafet_Casals@gmail.com,Septiembre,2023-09-18,Madrid,Apto,No Apto,Apto,Apto,Apto,DS,NaN,NaN,NaN,NaN,NaN
1,Jorge Manzanares,Jorge_Manzanares@gmail.com,Septiembre,2023-09-18,Madrid,Apto,No Apto,Apto,Apto,Apto,DS,NaN,NaN,NaN,NaN,NaN
2,Onofre Adadia,Onofre_Adadia@gmail.com,Septiembre,2023-09-18,Madrid,Apto,Apto,Apto,No Apto,Apto,DS,NaN,NaN,NaN,NaN,NaN


## 2. Tablas de catálogo

In [45]:
def make_lookup(values, id_col, nombre_col='nombre'):
    uniq = list(dict.fromkeys(values))  # preserva orden, elimina duplicados
    return pd.DataFrame({id_col: range(1, len(uniq) + 1), nombre_col: uniq})

campus    = make_lookup(sorted(all_est['Campus'].unique()),    'campus_id')
vertical  = make_lookup(['DS', 'FS'],                          'vertical_id')
promocion = make_lookup(all_est['Promocion'].unique(),         'promocion_id')
modalidad = make_lookup(claustro['Modalidad'].unique(),        'modalidad_id')
rol       = make_lookup(claustro['Rol'].unique(),              'rol_id')

for name, df in [('campus', campus), ('vertical', vertical),
                 ('promocion', promocion), ('modalidad', modalidad), ('rol', rol)]:
    print(f'\n--- {name} ---')
    display(df)


--- campus ---


,campus_id,nombre
0,1,Madrid
1,2,Valencia



--- vertical ---


,vertical_id,nombre
0,1,DS
1,2,FS



--- promocion ---


,promocion_id,nombre
0,1,Septiembre
1,2,Febrero



--- modalidad ---


,modalidad_id,nombre
0,1,Presencial
1,2,Online



--- rol ---


,rol_id,nombre
0,1,TA
1,2,LI


## 3. proyecto_tipo

In [46]:
proyecto_tipo = pd.DataFrame(
    [(p, 'DS') for p in DS_PROYECTOS] + [(p, 'FS') for p in FS_PROYECTOS],
    columns=['nombre', 'vert']
).merge(vertical, left_on='vert', right_on='nombre', suffixes=('', '_drop'))

proyecto_tipo = proyecto_tipo[['nombre', 'vertical_id']].reset_index(drop=True)
proyecto_tipo.insert(0, 'proyecto_tipo_id', range(1, len(proyecto_tipo) + 1))

display(proyecto_tipo)

,proyecto_tipo_id,nombre,vertical_id
0,1,HLF,1
1,2,EDA,1
2,3,BBDD,1
3,4,ML,1
4,5,Deployment,1
5,6,WebDev,2
6,7,FrontEnd,2
7,8,Backend,2
8,9,React,2
9,10,FullStack,2


## 4. grupo

Los grupos se derivan tanto de los CSV de estudiantes como de `claustro.csv`
(hay grupos asignados a profesores sin estudiantes en los CSV).

In [47]:
# Mapa promocion → fecha_inicio (para grupos que solo aparecen en claustro)
fecha_map = all_est.groupby('Promocion')['Fecha_comienzo'].first().to_dict()

grupos_est  = all_est[['Campus', 'Vertical', 'Promocion', 'Fecha_comienzo']].drop_duplicates()
grupos_prof = (
    claustro[['Campus', 'Vertical', 'Promocion']]
    .drop_duplicates()
    .assign(Fecha_comienzo=lambda df: df['Promocion'].map(fecha_map))
)

grupo = (
    pd.concat([grupos_est, grupos_prof], ignore_index=True)
    .drop_duplicates(subset=['Campus', 'Vertical', 'Promocion'])
    .merge(campus.rename(columns={'nombre': 'Campus'}),       on='Campus')
    .merge(vertical.rename(columns={'nombre': 'Vertical'}),   on='Vertical')
    .merge(promocion.rename(columns={'nombre': 'Promocion'}),  on='Promocion')
    .sort_values(['vertical_id', 'campus_id', 'promocion_id'])
    .reset_index(drop=True)
)
grupo.insert(0, 'grupo_id', range(1, len(grupo) + 1))
grupo = grupo[['grupo_id', 'campus_id', 'vertical_id', 'promocion_id', 'Fecha_comienzo']]
grupo = grupo.rename(columns={'Fecha_comienzo': 'fecha_inicio'})
grupo['fecha_inicio'] = grupo['fecha_inicio'].dt.strftime('%Y-%m-%d')

display(grupo)

,grupo_id,campus_id,vertical_id,promocion_id,fecha_inicio
0,1,1,1,1,2023-09-18
1,2,1,1,2,2024-02-12
2,3,1,2,1,2023-09-18
3,4,1,2,2,2024-02-12
4,5,2,2,1,2023-09-18
5,6,2,2,2,2024-02-12


## 5. estudiante

In [48]:
est_merged = (
    all_est
    .merge(campus.rename(columns={'nombre': 'Campus'}),      on='Campus')
    .merge(vertical.rename(columns={'nombre': 'Vertical'}),  on='Vertical')
    .merge(promocion.rename(columns={'nombre': 'Promocion'}), on='Promocion')
    .merge(
        grupo[['grupo_id', 'campus_id', 'vertical_id', 'promocion_id']],
        on=['campus_id', 'vertical_id', 'promocion_id']
    )
)

estudiante = (
    est_merged[['Nombre', 'Email', 'grupo_id']]
    .rename(columns={'Nombre': 'nombre', 'Email': 'email'})
    .reset_index(drop=True)
)
estudiante.insert(0, 'estudiante_id', range(1, len(estudiante) + 1))

display(estudiante)

,estudiante_id,nombre,email,grupo_id
0,1,Jafet Casals,Jafet_Casals@gmail.com,1
1,2,Jorge Manzanares,Jorge_Manzanares@gmail.com,1
2,3,Onofre Adadia,Onofre_Adadia@gmail.com,1
3,4,Merche Prada,Merche_Prada@gmail.com,1
4,5,Pilar Abella,Pilar_Abella@gmail.com,1
5,6,Leoncio Tena,Leoncio_Tena@gmail.com,1
6,7,Odalys Torrijos,Odalys_Torrijos@gmail.com,1
7,8,Eduardo Caparrós,Eduardo_Caparrós@gmail.com,1
8,9,Ignacio Goicoechea,Ignacio_Goicoechea@gmail.com,1
9,10,Clementina Santos,Clementina_Santos@gmail.com,1


## 6. calificacion

`melt` convierte las columnas de proyecto en filas.

In [49]:
partes = []
for vert, proyectos in [('DS', DS_PROYECTOS), ('FS', FS_PROYECTOS)]:
    subset = (
        est_merged[est_merged['Vertical'] == vert]
        .merge(estudiante[['estudiante_id', 'email']], left_on='Email', right_on='email')
    )
    partes.append(
        subset.melt(
            id_vars=['estudiante_id'],
            value_vars=proyectos,
            var_name='proyecto',
            value_name='resultado'
        )
    )

calificacion = (
    pd.concat(partes, ignore_index=True)
    .merge(proyecto_tipo.rename(columns={'nombre': 'proyecto'}), on='proyecto')
    .sort_values(['estudiante_id', 'proyecto_tipo_id'])
    [['estudiante_id', 'proyecto_tipo_id', 'resultado']]
    .reset_index(drop=True)
)
calificacion.insert(0, 'calificacion_id', range(1, len(calificacion) + 1))

display(calificacion)

,calificacion_id,estudiante_id,proyecto_tipo_id,resultado
0,1,1,1,Apto
1,2,1,2,No Apto
2,3,1,3,Apto
3,4,1,4,Apto
4,5,1,5,Apto
...,...,...,...,...
255,256,52,6,Apto
256,257,52,7,No Apto
257,258,52,8,No Apto
258,259,52,9,Apto


## 7. profesor

In [50]:
profesor = (
    claustro
    .merge(rol.rename(columns={'nombre': 'Rol'}), on='Rol')
    [['Nombre', 'rol_id']]
    .rename(columns={'Nombre': 'nombre'})
    .drop_duplicates('nombre')
    .reset_index(drop=True)
)
profesor.insert(0, 'profesor_id', range(1, len(profesor) + 1))

display(profesor)

,profesor_id,nombre,rol_id
0,1,Noa Yáñez,1
1,2,Saturnina Benitez,1
2,3,Anna Feliu,1
3,4,Rosalva Ayuso,1
4,5,Ana Sofía Ferrer,1
5,6,Angélica Corral,1
6,7,Ariel Lledó,1
7,8,Mario Prats,2
8,9,Luis Ángel Suárez,2
9,10,María Dolores Diaz,2


## 8. profesor_grupo

In [51]:
profesor_grupo = (
    claustro
    .merge(profesor.rename(columns={'nombre': 'Nombre'}),     on='Nombre')
    .merge(vertical.rename(columns={'nombre': 'Vertical'}),   on='Vertical')
    .merge(promocion.rename(columns={'nombre': 'Promocion'}),  on='Promocion')
    .merge(campus.rename(columns={'nombre': 'Campus'}),       on='Campus')
    .merge(
        grupo[['grupo_id', 'campus_id', 'vertical_id', 'promocion_id']],
        on=['campus_id', 'vertical_id', 'promocion_id']
    )
    .merge(modalidad.rename(columns={'nombre': 'Modalidad'}),  on='Modalidad')
    [['profesor_id', 'grupo_id', 'modalidad_id']]
    .sort_values(['grupo_id', 'profesor_id'])
    .reset_index(drop=True)
)
profesor_grupo.insert(0, 'profesor_grupo_id', range(1, len(profesor_grupo) + 1))

display(profesor_grupo)

,profesor_grupo_id,profesor_id,grupo_id,modalidad_id
0,1,1,1,1
1,2,2,1,1
2,3,7,1,1
3,4,10,1,2
4,5,3,3,1
5,6,9,3,2
6,7,6,4,1
7,8,4,5,1
8,9,5,6,1
9,10,8,6,2


## 9. Generar SQL 

In [52]:
import numpy as np

def fmt(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return 'NULL'
    if isinstance(v, (int, np.integer)): return str(int(v))
    return "'" + str(v).replace("'", "''") + "'"

def print_insert(tabla, df, cols):
    rows = ['    (' + ', '.join(fmt(row[c]) for c in cols) + ')' for _, row in df.iterrows()]
    print(f'INSERT INTO {tabla} ({", ".join(cols)}) VALUES')
    print(',\n'.join(rows) + ';\n')

secciones = [
    ('campus',         campus,         ['nombre']),
    ('vertical',       vertical,       ['nombre']),
    ('promocion',      promocion,      ['nombre']),
    ('modalidad',      modalidad,      ['nombre']),
    ('rol',            rol,            ['nombre']),
    ('proyecto_tipo',  proyecto_tipo,  ['nombre', 'vertical_id']),
    ('grupo',          grupo,          ['campus_id', 'vertical_id', 'promocion_id', 'fecha_inicio']),
    ('estudiante',     estudiante,     ['nombre', 'email', 'grupo_id']),
    ('calificacion',   calificacion,   ['estudiante_id', 'proyecto_tipo_id', 'resultado']),
    ('profesor',       profesor,       ['nombre', 'rol_id']),
    ('profesor_grupo', profesor_grupo, ['profesor_id', 'grupo_id', 'modalidad_id']),
]

for tabla, df, cols in secciones:
    print_insert(tabla, df, cols)


INSERT INTO campus (nombre) VALUES
    ('Madrid'),
    ('Valencia');

INSERT INTO vertical (nombre) VALUES
    ('DS'),
    ('FS');

INSERT INTO promocion (nombre) VALUES
    ('Septiembre'),
    ('Febrero');

INSERT INTO modalidad (nombre) VALUES
    ('Presencial'),
    ('Online');

INSERT INTO rol (nombre) VALUES
    ('TA'),
    ('LI');

INSERT INTO proyecto_tipo (nombre, vertical_id) VALUES
    ('HLF', 1),
    ('EDA', 1),
    ('BBDD', 1),
    ('ML', 1),
    ('Deployment', 1),
    ('WebDev', 2),
    ('FrontEnd', 2),
    ('Backend', 2),
    ('React', 2),
    ('FullStack', 2);

INSERT INTO grupo (campus_id, vertical_id, promocion_id, fecha_inicio) VALUES
    (1, 1, 1, '2023-09-18'),
    (1, 1, 2, '2024-02-12'),
    (1, 2, 1, '2023-09-18'),
    (1, 2, 2, '2024-02-12'),
    (2, 2, 1, '2023-09-18'),
    (2, 2, 2, '2024-02-12');

INSERT INTO estudiante (nombre, email, grupo_id) VALUES
    ('Jafet Casals', 'Jafet_Casals@gmail.com', 1),
    ('Jorge Manzanares', 'Jorge_Manzanares@gmail.com', 1)